In [4]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import glob
from pathlib import Path


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/anaconda3/envs/gds_environment/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/anaconda3/envs/gds_environment/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/local/anaconda3/envs/gds_environment/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 7

Here is the code to generate the Neural network model. This code has been run on kaggle, and will not work in notebooks. At the bottom, we load the model, which has been generated in kaggle. Here we use the weights to predict if an article is fake or real.

In [12]:

files = sorted(Path("processed_chunks_parquet").glob("chunk_*.parquet"))



print("Done!")
print(files)

Done!
[PosixPath('processed_chunks_parquet/chunk_000.parquet'), PosixPath('processed_chunks_parquet/chunk_001.parquet'), PosixPath('processed_chunks_parquet/chunk_002.parquet'), PosixPath('processed_chunks_parquet/chunk_003.parquet'), PosixPath('processed_chunks_parquet/chunk_004.parquet'), PosixPath('processed_chunks_parquet/chunk_005.parquet'), PosixPath('processed_chunks_parquet/chunk_006.parquet'), PosixPath('processed_chunks_parquet/chunk_007.parquet'), PosixPath('processed_chunks_parquet/chunk_008.parquet'), PosixPath('processed_chunks_parquet/chunk_009.parquet'), PosixPath('processed_chunks_parquet/chunk_010.parquet'), PosixPath('processed_chunks_parquet/chunk_011.parquet'), PosixPath('processed_chunks_parquet/chunk_012.parquet'), PosixPath('processed_chunks_parquet/chunk_013.parquet'), PosixPath('processed_chunks_parquet/chunk_014.parquet'), PosixPath('processed_chunks_parquet/chunk_015.parquet'), PosixPath('processed_chunks_parquet/chunk_016.parquet'), PosixPath('processed_chu

In [14]:
from sklearn.model_selection import train_test_split

fake_categories = {"fake", "hate", "unreliable", "clickbait", "conspiracy", "junksci", "rumor"}
valid_types = fake_categories | {"reliable", "political", "bias"}

def load_chunks_logistic(files):
    dfs = []
    for file in tqdm(files):
        df = pd.read_parquet(file)
        df = df[df["type"].isin(valid_types)][["type", "tokens_stemmed", "domain"]]
        dfs.append(df)
    data = pd.concat(dfs, ignore_index=True)
    del dfs
    data["text"]            = data["tokens_stemmed"].apply(lambda tokens: " ".join(tokens))
    data["text_and_domain"] = data["tokens_stemmed"].apply(lambda tokens: " ".join(tokens)) + " " + data["domain"]
    data["label"]           = data["type"].apply(lambda x: 1 if x in fake_categories else 0)
    return data[["text", "text_and_domain", "label"]]

all_data = load_chunks_logistic(files)
print(f"Total rows: {len(all_data)}")
print(f"Label distribution:\n{all_data['label'].value_counts()}")


train_data, temp_data = train_test_split(all_data, test_size=0.2, random_state=42)
val_data,  test_data  = train_test_split(temp_data, test_size=0.5, random_state=42)

X_train, y_train = train_data["text"], train_data["label"]
X_val,   y_val   = val_data["text"],   val_data["label"]
X_test,  y_test  = test_data["text"],  test_data["label"]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")



  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:30<00:00,  3.28it/s]


Total rows: 890518
Label distribution:
label
0    546313
1    344205
Name: count, dtype: int64
Train: 712414, Val: 89052, Test: 89052


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

N_FEATURES = 10000

vectorizer = TfidfVectorizer(max_features=N_FEATURES)
X_train_vec = vectorizer.fit_transform(X_train) 
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)



In [23]:
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import Dataset

class SparseDataset(Dataset):
    def __init__(self, X_sparse, y):
        self.X = X_sparse
        self.y = torch.tensor(y.values, dtype=torch.long)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        
        x = torch.tensor(self.X[idx].toarray().squeeze(), dtype=torch.float32)
        return x, self.y[idx]
    

batch_size = 256

train_loader = DataLoader(SparseDataset(X_train_vec, y_train), 
                          batch_size=256, shuffle=True, num_workers=0)
val_loader   = DataLoader(SparseDataset(X_val_vec, y_val),   
                          batch_size=256, num_workers=0)
test_loader  = DataLoader(SparseDataset(X_test_vec, y_test),  
                          batch_size=256, num_workers=0)


In [ ]:
# This is to test Liarset

import joblib

liar_raw = pd.read_csv("liar_dataset/test.tsv", sep="\t", header=None)

liar_raw.columns = [
    "id",
    "label",
    "statement",
    "subject",
    "speaker",
    "speaker_job_title",
    "state_info",
    "party_affiliation",
    "barely_true_counts",
    "false_counts",
    "half_true_counts",
    "mostly_true_counts",
    "pants_fire_counts",
    "context"
]


print(liar_raw.head())
print(liar_raw["label"].value_counts())
#Samme kategorier tværs modellerne for kontinuitet og mere målbare resultater 
fake_categories_liar = {"false", "pants-fire", "barely-true"}
valid_types_liar = fake_categories_liar | {"true", "mostly-true"}

# Group mapping
fake_categories_liar = {"false", "pants-fire", "barely-true"}
real_categories_liar = {"true", "mostly-true"}
valid_categories_liar = fake_categories_liar | real_categories_liar

# Find tekstkolonnen
TEXT_COL = "text"
if TEXT_COL not in liar_raw.columns:
    TEXT_COL = "statement"

# Lav clean liar_df
liar_df = liar_raw.copy()

# Behold kun de labels vi vil bruge
liar_df = liar_df[liar_df["label"].isin(valid_categories_liar)].copy()

# Fjern manglende/blank tekst
liar_df = liar_df[liar_df[TEXT_COL].notna()].copy()
liar_df[TEXT_COL] = liar_df[TEXT_COL].astype(str).str.strip()
liar_df = liar_df[liar_df[TEXT_COL] != ""].copy()

# Preprocess LIAR text
liar_df["tokens"] = liar_df[TEXT_COL].apply(tokenize_text)
liar_df["tokens_clean"] = liar_df["tokens"].apply(clean_tokens)
liar_df["tokens_stemmed"] = liar_df["tokens_clean"].apply(stem_tokens)
liar_df["text"] = liar_df["tokens_stemmed"].apply(lambda tokens: " ".join(tokens))

liar_df["target"] = liar_df["label"].apply(
    lambda x: 0 if x in fake_categories_liar else 1
)

# Load the vectorizer 
vectorizer = joblib.load("NN_Model/vectorizer.pkl")

# Apply it to the liar data set
liar_test_vec = vectorizer.transform(liar_df["text"])

# Overwrite the test_loader
test_loader_liar  = DataLoader(SparseDataset(liar_test_vec, liar_df["target"]),  
                          batch_size=256, num_workers=0)



           id       label                                          statement  \
0  11972.json        true  Building a wall on the U.S.-Mexico border will...   
1  11685.json       false  Wisconsin is on pace to double the number of l...   
2  11096.json       false  Says John McCain has done nothing to help the ...   
3   5209.json   half-true  Suzanne Bonamici supports a plan that will cut...   
4   9524.json  pants-fire  When asked by a reporter whether hes at the ce...   

                                             subject  \
0                                        immigration   
1                                               jobs   
2                    military,veterans,voting-record   
3  medicare,message-machine-2012,campaign-adverti...   
4  campaign-finance,legal-issues,campaign-adverti...   

                            speaker     speaker_job_title state_info  \
0                        rick-perry              Governor      Texas   
1                 katrina-shankland  S

/usr/local/anaconda3/envs/gds_environment/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/anaconda3/envs/gds_environment/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

class Model(nn.Module):
    def __init__(self, in_features=10000, h1=512, h2=256, h3=128, output=2):
        super().__init__()
        self.fc1   = nn.Linear(in_features, h1)
        self.drop1 = nn.Dropout(0.5)
        self.fc2   = nn.Linear(h1, h2)
        self.drop2 = nn.Dropout(0.5)
        self.fc3   = nn.Linear(h2, h3)
        self.drop3 = nn.Dropout(0.5)
        self.out   = nn.Linear(h3, output)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.drop1(x)
        x = F.relu(self.fc2(x))
        x = self.drop2(x)
        x = F.relu(self.fc3(x))
        x = self.drop3(x)
        x = self.out(x)
        return x


In [ ]:
model     = Model(in_features=N_FEATURES).to(device)
# Lower starting learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

# Add scheduler — halves lr when val loss stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=2, factor=0.5
)
criterion = nn.CrossEntropyLoss()

print("Done")

In [ ]:
import pickle
with open("/kaggle/working/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss, train_correct = 0, 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        train_correct += (preds.argmax(dim=1) == y_batch).sum().item()

    train_acc  = train_correct / len(train_loader.dataset)
    train_loss = train_loss / len(train_loader)

    # --- Validate ---
    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            preds        = model(X_batch)
            val_loss    += criterion(preds, y_batch).item()
            val_correct += (preds.argmax(dim=1) == y_batch).sum().item()

    val_acc  = val_correct / len(val_loader.dataset)
    val_loss = val_loss / len(val_loader)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

<bound method Module.parameters of Model(
  (fc1): Linear(in_features=3000, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (out): Linear(in_features=128, out_features=2, bias=True)
)>

In [ ]:
model.eval()
test_loss, test_correct = 0, 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)  # move to GPU
        y_batch = y_batch.to(device)  # move to GPU

        preds = model(X_batch)
        test_loss    += criterion(preds, y_batch).item()
        test_correct += (preds.argmax(dim=1) == y_batch).sum().item()

test_acc  = test_correct / len(test_loader.dataset)
test_loss = test_loss / len(test_loader)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

In [ ]:
torch.save(model.state_dict(), "/kaggle/working/model3.pth")

In [21]:
# We load the model
model = Model(in_features=10000, h1=512, h2=256, h3=128, output=2)

# Load the saved weights
model.load_state_dict(torch.load("NN_Model/model3.pth", map_location="cpu"))
model.eval()

Model(
  (fc1): Linear(in_features=10000, out_features=512, bias=True)
  (drop1): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (drop2): Dropout(p=0.5, inplace=False)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (drop3): Dropout(p=0.5, inplace=False)
  (out): Linear(in_features=128, out_features=2, bias=True)
)

In [24]:
# For fake news set 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = model(X_batch).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(y_batch.tolist())

print(f"Test Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(classification_report(all_labels, all_preds, target_names=["REAL", "FAKE"]))
print(confusion_matrix(all_labels, all_preds))

Test Accuracy: 0.9064
              precision    recall  f1-score   support

        REAL       0.92      0.93      0.92     54667
        FAKE       0.89      0.86      0.88     34385

    accuracy                           0.91     89052
   macro avg       0.90      0.90      0.90     89052
weighted avg       0.91      0.91      0.91     89052

[[51004  3663]
 [ 4674 29711]]


In [ ]:
# For liar set 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader_liar:
        preds = model(X_batch).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(y_batch.tolist())

print(f"Test Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(classification_report(all_labels, all_preds, target_names=["REAL", "FAKE"]))
print(confusion_matrix(all_labels, all_preds))

Test Accuracy: 0.5250
              precision    recall  f1-score   support

        REAL       0.56      0.65      0.60       553
        FAKE       0.46      0.38      0.42       449

    accuracy                           0.52      1002
   macro avg       0.51      0.51      0.51      1002
weighted avg       0.52      0.52      0.52      1002

[[357 196]
 [280 169]]
